In [4]:
from ipywidgets import interact

import gplately

from lib.main import *
from lib.plot import *

from parameters import parameters

In [5]:
# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
#time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)
time_steps = [44,67,100,169,170,180,245,290,330,406,433,443,463,475,500,555,715,745,843,1350]

buffer_distance = parameters["buffer_distance"]

plate_model_dir = "../feature_extraction_stellar4a/plate_model"
outputs_dir = parameters["outputs_dir"]
buffer_zones_dir = parameters["buffer_zones_dir"]
buffer_zone_maps_dir = parameters["buffer_zone_maps_dir"]

buffer_zones_dir = os.path.join(outputs_dir, buffer_zones_dir)
buffer_zone_maps_dir = os.path.join(outputs_dir, buffer_zone_maps_dir)

nprocs = 12

In [6]:
rotation_model = [
    plate_model_dir+"/CombinedRotations.rot",
]

topology_features = [
    plate_model_dir+"/Feature_Geometries.gpml",
    plate_model_dir+"/Flat_Slabs.gpml",
    plate_model_dir+"/Plate_Boundaries.gpml",
]

#static_polygons = plate_model_dir+"/static_polygons.gpmlz"

plate_model = gplately.PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features
    #static_polygons=static_polygons
)

coastlines_filename = "StaticGeometries/Coastlines/Global_coastlines_low_res.shp"
coastlines = os.path.join(plate_model_dir, coastlines_filename)
continents_filename = "StaticGeometries/ContinentalPolygons/Global_EarthByte_GPlates_PresentDay_ContinentsAndArcs.shp"
continents = os.path.join(plate_model_dir, continents_filename)
COBs_filename = "StaticGeometries/COBLineSegments/Global_EarthByte_GeeK07_COBLineSegments_2019_v1.shp"
COBs = os.path.join(plate_model_dir, COBs_filename)

gplot = gplately.PlotTopologies(plate_model, continents=continents, COBs=COBs, coastlines=coastlines)

projection = ccrs.Mollweide(central_longitude=60)

In [9]:
if not os.path.isdir(buffer_zones_dir):
    run_create_buffer_zones(
        nprocs=nprocs,
        times=time_steps,
        topological_features=topology_features,
        rotation_model=rotation_model,
        output_dir=buffer_zones_dir,
        buffer_distance=buffer_distance,
        verbose=True,
        return_output=False,
    )

Output directory does not exist; creating now: outputs\buffer_zones
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:   12.6s remaining:  1.1min
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:   14.7s finished


In [10]:
@interact
def show_map(time=time_steps):
    gplot.time = time
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")
    ax.set_global()

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor='palegreen',
        edgecolor='none',
        alpha=0.7,
        zorder=4,
    )

    gplot.plot_all_topologies(ax, color='orangered', zorder=5)
    gplot.plot_trenches(ax, color='black', zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='black', zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='tan', edgecolor='none', label='Continental Crust'),
        Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color="orangered", label="Mid-Ocean Ridges"),
        Line2D([0], [0], color="black", label="Trench Lines")
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.2))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
    
    plt.show()

interactive(children=(Dropdown(description='time', options=(44, 67, 100, 169, 170, 180, 245, 290, 330, 406, 43…

In [ ]:
for time in time_steps:
    gplot.time = time
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")
    ax.set_global()

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor='palegreen',
        edgecolor='none',
        alpha=0.7,
        zorder=4,
    )

    gplot.plot_all_topologies(ax, color='orangered', zorder=5)
    gplot.plot_trenches(ax, color='black', zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='black', zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='tan', edgecolor='none', label='Continental Crust'),
        Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color="orangered", label="Mid-Ocean Ridges"),
        Line2D([0], [0], color="black", label="Trench Lines")
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.2))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)

    filename = os.path.join(buffer_zone_maps_dir, f"buffer_zone_map_{time:.0f}Ma.png")
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.close()


In [6]:
if os.path.exists(buffer_zone_maps_dir):
    print(f"Buffer zone maps are located in {buffer_zone_maps_dir}")
else:
    os.makedirs(buffer_zone_maps_dir, exist_ok=True)
    
    generate_buffer_zone_maps(
        rotation_model,
        topology_features,
        static_polygons,
        coastlines,
        continents,
        COBs,
        projection,
        time_steps,
        buffer_zones_dir,
        output_dir=buffer_zone_maps_dir,
        n_jobs=nprocs
    )
    
    output_filenames = [
        os.path.join(buffer_zone_maps_dir, f"buffer_zone_map_{t:0.0f}Ma.png")
        for t in time_steps
    ]
    
    output_filename = os.path.join(outputs_dir, "buffer_zone_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

Buffer zone maps are located in outputs\buffer_zone_maps
